# 10 — PCA Benchmark + Rolling Time-Series Cross-Validation

1. ***PCA ** is applied only to the **weather block** (and weather lags), not to the full feature space.
2. Here we switch from one fixed validation split to **rolling / expanding time-series CV** on the development period.

**What this notebook does**
- uses the same processed file path as your earlier notebooks: `../data/processed/supervised_hood_3h_multiclass.csv`
- keeps **2025-07-01 onward** as the untouched final test period
- runs rolling CV on the development period (**2023-01-01 to 2025-06-30**)
- compares two benchmark models:
  - **Logistic Regression without PCA**
  - **Logistic Regression with PCA only on weather-related columns**
- reports **argmax** results only (no threshold tuning here)

**Interpretation rule**
- If PCA improves CV results, keep it as a benchmark answer 
- If PCA does not help, you can still confidently say that you tested it properly and it was not beneficial for your current feature/model design.


In [27]:

from __future__ import annotations

from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    recall_score,
    precision_score,
    average_precision_score,
    mean_squared_error,
)

warnings.filterwarnings("ignore")

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SUPERVISED_PATH = DATA_DIR / "supervised_hood_3h_multiclass.csv"
DEV_END = pd.Timestamp("2025-06-30 23:59:59")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

PCA_VARIANCE = 0.95
MIN_TRAIN_DAYS = 365
VAL_DAYS = 90
STEP_DAYS = 90
MAX_FOLDS = 4

OUT_CV_SUMMARY = MODEL_DIR / "pca_cv_summary.csv"
OUT_TEST = MODEL_DIR / "pca_final_summary.csv"

print("Dataset path:", SUPERVISED_PATH)
print("Exists:", SUPERVISED_PATH.exists())


Dataset path: ../data/processed/supervised_hood_3h_multiclass.csv
Exists: True


In [28]:

df = pd.read_csv(SUPERVISED_PATH, low_memory=False)
df["time_3h"] = pd.to_datetime(df["time_3h"], errors="coerce")
df["HOOD_158_CODE"] = df["HOOD_158_CODE"].astype(str).str.zfill(3)
df["y_class"] = pd.to_numeric(df["y_class"], errors="coerce")
df = df.dropna(subset=["time_3h", "y_class"]).copy()
df["y_class"] = df["y_class"].astype("int8")
df = df.sort_values(["time_3h", "HOOD_158_CODE"]).reset_index(drop=True)

dev_mask = df["time_3h"] <= DEV_END
test_mask = df["time_3h"] > DEV_END

feature_cols = [c for c in df.columns if c not in ["time_3h", "y_class", "y_count_next"]]
X_dev = df.loc[dev_mask, feature_cols].reset_index(drop=True)
y_dev = df.loc[dev_mask, "y_class"].reset_index(drop=True)
t_dev = df.loc[dev_mask, "time_3h"].reset_index(drop=True)

X_test = df.loc[test_mask, feature_cols].reset_index(drop=True)
y_test = df.loc[test_mask, "y_class"].reset_index(drop=True)
test_times = df.loc[test_mask, "time_3h"].reset_index(drop=True)

print("Development shape:", X_dev.shape)
print("Test shape:", X_test.shape)
print("Development class dist:")
print(y_dev.value_counts(normalize=True).sort_index().round(4))
print("Test class dist:")
print(y_test.value_counts(normalize=True).sort_index().round(4))


Development shape: (1151504, 47)
Test shape: (232418, 47)
Development class dist:
y_class
0    0.8927
1    0.0930
2    0.0143
Name: proportion, dtype: float64
Test class dist:
y_class
0    0.8942
1    0.0912
2    0.0147
Name: proportion, dtype: float64


In [29]:
def metric_dict(y_true, pred, proba):
    y_true_arr = np.asarray(y_true)
    y_pred_arr = np.asarray(pred)
    proba_arr = np.asarray(proba)

    rec = recall_score(
        y_true_arr, y_pred_arr,
        labels=[0, 1, 2],
        average=None,
        zero_division=0
    )
    macro_rec = float(np.mean(rec))

    prec2 = precision_score(
        (y_true_arr == 2).astype(int),
        (y_pred_arr == 2).astype(int),
        zero_division=0
    )
    pred_class2_rate = float((y_pred_arr == 2).mean())

    # AP / MAP
    ap_vals = []
    class_support = []
    ap_dict = {}

    for cls in [0, 1, 2]:
        y_true_bin = (y_true_arr == cls).astype(int)
        support = int(y_true_bin.sum())
        class_support.append(support)

        if support == 0:
            ap = np.nan
        else:
            ap = float(average_precision_score(y_true_bin, proba_arr[:, cls]))

        ap_dict[f"ap_class{cls}"] = ap
        if not np.isnan(ap):
            ap_vals.append(ap)

    macro_map = float(np.mean(ap_vals)) if len(ap_vals) else np.nan

    valid_pairs = [
        (ap_dict[f"ap_class{cls}"], class_support[cls])
        for cls in [0, 1, 2]
        if not np.isnan(ap_dict[f"ap_class{cls}"])
    ]
    if len(valid_pairs):
        weighted_map = float(
            np.average(
                [x[0] for x in valid_pairs],
                weights=[x[1] for x in valid_pairs]
            )
        )
    else:
        weighted_map = np.nan

    label_rmse = float(np.sqrt(mean_squared_error(y_true_arr, y_pred_arr)))

    y_onehot = np.zeros((len(y_true_arr), 3), dtype=float)
    y_onehot[np.arange(len(y_true_arr)), y_true_arr.astype(int)] = 1.0
    prob_rmse_macro = float(
        np.mean([
            np.sqrt(mean_squared_error(y_onehot[:, c], proba_arr[:, c]))
            for c in range(3)
        ])
    )

    row = {
        "macro_recall": macro_rec,
        "recall_0": float(rec[0]),
        "recall_1": float(rec[1]),
        "recall_2": float(rec[2]),
        "precision_2": float(prec2),
        "pred_class2_rate": pred_class2_rate,
        "macro_map": macro_map,
        "weighted_map": weighted_map,
        "label_rmse": label_rmse,
        "prob_rmse_macro": prob_rmse_macro,
    }
    row.update(ap_dict)
    return row


def print_metrics(y_true, y_pred, proba=None, label="Model"):
    m = metric_dict(y_true, y_pred, proba=proba)
    m["label"] = label
    display(pd.DataFrame([m]))
    return m

In [30]:
# ============================================================
# Column groups for preprocessing / PCA
# ============================================================
import pandas as pd

# make sure these special columns are excluded from features if present
exclude_cols = {"y_class", "target", "time_3h"}

# start from X_dev columns, because that is what CV uses
all_feature_cols = list(X_dev.columns)

# categorical columns: keep only the ones that actually exist
candidate_cat_cols = ["HOOD_158_CODE"]
cat_cols = [c for c in candidate_cat_cols if c in all_feature_cols]

# numeric columns = everything else
num_cols = [c for c in all_feature_cols if c not in cat_cols and c not in exclude_cols]

# weather columns for PCA: keep only numeric weather-like columns that exist
weather_keywords = [
    "temp", "temperature", "dew", "humidity", "wind", "pressure",
    "visibility", "rain", "snow", "precip", "weather", "storm",
    "gust", "cloud", "humidex", "windchill"
]

weather_cols = [
    c for c in num_cols
    if any(k in c.lower() for k in weather_keywords)
]

# fallback: if weather_cols becomes empty, use all numeric columns
if len(weather_cols) == 0:
    weather_cols = num_cols.copy()

print("Total features:", len(all_feature_cols))
print("Categorical cols:", cat_cols)
print("Numeric cols:", len(num_cols))
print("Weather cols for PCA:", len(weather_cols))
print(weather_cols[:20])

Total features: 47
Categorical cols: ['HOOD_158_CODE']
Numeric cols: 46
Weather cols for PCA: 18
['pressure_sea', 'wind_speed', 'relative_humidity', 'temperature', 'cloud_cover_8', 'rain', 'snow', 'visibility', 'snow_on_ground', 'temperature_lag_1', 'visibility_lag_1', 'rain_lag_1', 'snow_lag_1', 'snow_on_ground_lag_1', 'wind_speed_lag_1', 'relative_humidity_lag_1', 'pressure_sea_lag_1', 'cloud_cover_8_lag_1']


In [31]:

def build_logreg_no_pca():
    preprocess = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                num_cols,
            ),
            (
                "cat",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]),
                cat_cols,
            ),
        ],
        remainder="drop",
    )

    clf = LogisticRegression(
        solver="saga",
        max_iter=1200,
        tol=1e-3,
        class_weight="balanced",
        multi_class="multinomial",
        n_jobs=-1,
        random_state=RANDOM_SEED,
    )
    return Pipeline(steps=[("preprocess", preprocess), ("clf", clf)])


def build_logreg_weather_pca():
    non_weather_num_cols = [c for c in num_cols if c not in weather_cols]

    preprocess = ColumnTransformer(
        transformers=[
            (
                "weather_pca",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                    ("pca", PCA(n_components=0.95, random_state=RANDOM_SEED)),
                ]),
                weather_cols
            ),
            (
                "num_other",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                non_weather_num_cols
            ),
            (
                "cat",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]),
                cat_cols
            ),
        ],
        remainder="drop"
    )

    clf = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        multi_class="multinomial",
        random_state=RANDOM_SEED,
    )

    return Pipeline(steps=[("preprocess", preprocess), ("clf", clf)])

def run_cv(model_name, builder):
    rows = []
    for fold_id, f in enumerate(folds, start=1):
        tr_mask = (t_dev >= f["train_start"]) & (t_dev <= f["train_end"])
        va_mask = (t_dev >= f["val_start"]) & (t_dev <= f["val_end"])

        Xtr, ytr = X_dev.loc[tr_mask].copy(), y_dev.loc[tr_mask].copy()
        Xva, yva = X_dev.loc[va_mask].copy(), y_dev.loc[va_mask].copy()

        model = builder()
        model.fit(Xtr, ytr)
        pred = model.predict(Xva)
        proba = model.predict_proba(Xva)

        row = metric_dict(yva, pred, proba)
        row["model"] = model_name
        row["fold"] = fold_id
        rows.append(row)
        
        print(f"Done: {model_name} | fold {fold_id}/{len(folds)} | macro={row['macro_recall']:.4f} | ap2={row.get('ap_class2', np.nan):.4f}")
    return pd.DataFrame(rows)


In [33]:
# ============================================================
# Build rolling time-series CV folds
# ============================================================
import numpy as np
import pandas as pd

# make sure t_dev is datetime
t_dev = pd.to_datetime(t_dev)

# unique ordered timestamps in development set
unique_times = np.array(sorted(pd.Series(t_dev).dropna().unique()))
print("Unique dev timestamps:", len(unique_times))

N_SPLITS = 3
MIN_TRAIN_FRAC = 0.55
VAL_FRAC = 0.15

n_times = len(unique_times)
val_size = max(1, int(n_times * VAL_FRAC))
start_train_end = max(2, int(n_times * MIN_TRAIN_FRAC))

folds = []
for i in range(N_SPLITS):
    train_end_idx = start_train_end + i * val_size
    val_start_idx = train_end_idx
    val_end_idx = min(train_end_idx + val_size - 1, n_times - 1)

    if val_start_idx >= n_times or val_start_idx > val_end_idx:
        break

    folds.append({
        "train_start": unique_times[0],
        "train_end": unique_times[train_end_idx - 1],
        "val_start": unique_times[val_start_idx],
        "val_end": unique_times[val_end_idx],
    })

print("Number of folds:", len(folds))
display(pd.DataFrame(folds))
cv_no_pca = run_cv("LogReg_no_PCA", build_logreg_no_pca)
cv_pca = run_cv("LogReg_weather_PCA", build_logreg_weather_pca)
cv_rows = pd.concat([cv_no_pca, cv_pca], ignore_index=True)

print(cv_rows.columns.tolist())

cv_summary = (
    cv_rows.groupby("model", as_index=False)
    .agg(
        folds=("fold", "count"),
        macro_recall_mean=("macro_recall", "mean"),
        macro_recall_std=("macro_recall", "std"),
        recall_1_mean=("recall_1", "mean"),
        recall_2_mean=("recall_2", "mean"),
        precision_2_mean=("precision_2", "mean"),
        pred_class2_rate_mean=("pred_class2_rate", "mean"),
        ap_class2_mean=("ap_class2", "mean"),
        macro_map_mean=("macro_map", "mean"),
        weighted_map_mean=("weighted_map", "mean"),
        label_rmse_mean=("label_rmse", "mean"),
        prob_rmse_macro_mean=("prob_rmse_macro", "mean"),
    )
    .sort_values(
        ["macro_recall_mean", "weighted_map_mean", "ap_class2_mean"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

display(cv_summary)

Unique dev timestamps: 7288
Number of folds: 3


,train_start,train_end,val_start,val_end
0,2023-01-02,2024-05-16 21:00:00,2024-05-17 00:00:00,2024-09-30 12:00:00
1,2023-01-02,2024-09-30 12:00:00,2024-09-30 15:00:00,2025-02-14 03:00:00
2,2023-01-02,2025-02-14 03:00:00,2025-02-14 06:00:00,2025-06-30 18:00:00


Done: LogReg_no_PCA | fold 1/3 | macro=0.5468 | ap2=0.1284
Done: LogReg_no_PCA | fold 2/3 | macro=0.5443 | ap2=0.1079
Done: LogReg_no_PCA | fold 3/3 | macro=0.5368 | ap2=0.1042
Done: LogReg_weather_PCA | fold 1/3 | macro=0.5469 | ap2=0.1304
Done: LogReg_weather_PCA | fold 2/3 | macro=0.5440 | ap2=0.1076
Done: LogReg_weather_PCA | fold 3/3 | macro=0.5340 | ap2=0.1035
['macro_recall', 'recall_0', 'recall_1', 'recall_2', 'precision_2', 'pred_class2_rate', 'macro_map', 'weighted_map', 'label_rmse', 'prob_rmse_macro', 'ap_class0', 'ap_class1', 'ap_class2', 'model', 'fold']


,model,folds,macro_recall_mean,macro_recall_std,recall_1_mean,recall_2_mean,precision_2_mean,pred_class2_rate_mean,ap_class2_mean,macro_map_mean,weighted_map_mean,label_rmse_mean,prob_rmse_macro_mean
0,LogReg_no_PCA,3,0.542623,0.005223,0.347392,0.635617,0.063616,0.147417,0.113543,0.399796,0.867433,0.838093,0.391964
1,LogReg_weather_PCA,3,0.541645,0.006752,0.344891,0.632786,0.063704,0.146564,0.113847,0.399952,0.867397,0.835553,0.391526


In [34]:
# ============================================================
# Final test comparison: evaluate BOTH no-PCA and PCA models
# ============================================================
final_test_rows = []

final_candidates = [
    ("LogReg_no_PCA", build_logreg_no_pca),
    ("LogReg_weather_PCA", build_logreg_weather_pca),
]

for model_name, builder in final_candidates:
    print(f"Fitting final {model_name} on full development data...")
    model = builder()
    model.fit(X_dev, y_dev)

    test_pred = model.predict(X_test)
    test_proba = model.predict_proba(X_test)

    row = metric_dict(y_test, test_pred, test_proba)
    row["model"] = f"10 {model_name} - Final Test"
    final_test_rows.append(row)

test_df = pd.DataFrame(final_test_rows)

# Keep compact columns for Notebook 15
test_df = test_df[
    [c for c in [
        "model",
        "macro_recall",
        "recall_2",
        "precision_2",
        "ap_class2",
        "macro_map",
        "prob_rmse_macro"
    ] if c in test_df.columns]
].copy()

test_df = test_df.sort_values(
    ["macro_recall", "ap_class2", "macro_map", "recall_2", "prob_rmse_macro"],
    ascending=[False, False, False, False, True]
).reset_index(drop=True)

# Save both final test rows
test_df.to_csv(OUT_TEST, index=False)

# Also save the CV summary in compact form
cv_summary_to_save = cv_summary.rename(columns={"label": "model"}).copy()
cv_summary_to_save.to_csv(OUT_CV_SUMMARY, index=False)

print("Saved:", OUT_CV_SUMMARY)
print("Saved:", OUT_TEST)
display(test_df)

Fitting final LogReg_no_PCA on full development data...
Fitting final LogReg_weather_PCA on full development data...
Saved: ../models/pca_cv_summary.csv
Saved: ../models/pca_final_summary.csv


,model,macro_recall,recall_2,precision_2,ap_class2,macro_map,prob_rmse_macro
0,10 LogReg_weather_PCA - Final Test,0.548238,0.668426,0.062329,0.117397,0.399952,0.395999
1,10 LogReg_no_PCA - Final Test,0.548016,0.668426,0.062070,0.117930,0.400086,0.395891
